# Multi search Agent

In [14]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

load_dotenv()
llm=ChatOpenAI()

from langsmith import Client
client = Client()
prompt = client.pull_prompt("hwchase17/react")


In [3]:
#Retrieve data  from wikipedia
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper=WikipediaAPIWrapper(top_k_results=1,doc_content_chars_max=300)
tool=WikipediaQueryRun(api_wrapper=api_wrapper)
tool.name

'wikipedia'

In [26]:
#Use website as source
loader=WebBaseLoader("https://docs.langchain.com/langsmith/home")
docs=loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)
embeddings = OpenAIEmbeddings()
vecterdb=FAISS.from_documents(documents,embeddings)
retriever=vecterdb.as_retriever()
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002619CAB9E50>, search_kwargs={})

In [5]:
from langchain_core.tools import create_retriever_tool
retreiver_tool=create_retriever_tool(retriever,'langsmith_search',"Search for information about the langsmith.For any question about langsmith this tool should be used")


In [6]:
#For Arxiv source
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun
arxiv_wrapper=ArxivAPIWrapper(top_k_results=1,doc_content_chars_max=200)
arxiv=ArxivQueryRun(arxiv_wrapper=arxiv_wrapper)
arxiv.name

'arxiv'

In [7]:
#Combining all the tools
tools=[tool,arxiv,retreiver_tool]

In [16]:
from langchain_classic.agents import AgentExecutor, create_react_agent
agent = create_react_agent(llm, tools, prompt)
agent_executor=AgentExecutor(agent=agent,tools=tools,verbose=True)

In [23]:
query="Tell me about Langsmith"
retrieved_docs=retriever.invoke(query)
contexts = [doc.page_content for doc in retrieved_docs]
response=agent_executor.invoke({"input":query})
response
answer=response["output"]



> Entering new AgentExecutor chain...
I should use the langsmith_search tool to find information about Langsmith.
Action: langsmith_search
Action Input: 'Langsmith'LangSmith docs - Docs by LangChainSkip to main contentJoin us May 13th & May 14th at Interrupt, the Agent Conference by LangChain. Buy tickets >Docs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyIntegrationsPlansEnterprise featuresAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APIAudit logsToolsPolly AI assistantCLISkillsSandboxesPrivate previewAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy pageLangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate

In [24]:
#Create evaluation dataset
from datasets import Dataset
data={
    "question":[query],
    "answer": [answer],
    "contexts": [contexts],
    "ground_truth": ["LangSmith is a platform for debugging, testing, and monitoring LLM applications."]
}
dataset = Dataset.from_dict(data)

In [27]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

result = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    embeddings=embeddings
)

print(result)

C:\Users\shres\AppData\Local\Temp\ipykernel_14012\657664223.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import (
C:\Users\shres\AppData\Local\Temp\ipykernel_14012\657664223.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import (
C:\Users\shres\AppData\Local\Temp\ipykernel_14012\657664223.py:2: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import (
C:\Users\shres\AppData\Local\Temp\ipykernel_14012

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'faithfulness': 1.0000, 'answer_relevancy': 0.9246, 'context_precision': 1.0000, 'context_recall': 1.0000}
